In [1]:
!unzip FixerX_deepseekmath-r1.zip

Archive:  FixerX_deepseekmath-r1.zip
   creating: math-meme-corrector/
  inflating: math-meme-corrector/README.md  
  inflating: math-meme-corrector/adapter_model.safetensors  
  inflating: math-meme-corrector/tokenizer_config.json  
  inflating: math-meme-corrector/tokenizer.json  
  inflating: math-meme-corrector/special_tokens_map.json  
  inflating: math-meme-corrector/adapter_config.json  
   creating: math-meme-corrector/checkpoint-595/
  inflating: math-meme-corrector/checkpoint-595/training_args.bin  
  inflating: math-meme-corrector/checkpoint-595/README.md  
  inflating: math-meme-corrector/checkpoint-595/scheduler.pt  
  inflating: math-meme-corrector/checkpoint-595/adapter_model.safetensors  
  inflating: math-meme-corrector/checkpoint-595/optimizer.pt  
  inflating: math-meme-corrector/checkpoint-595/rng_state.pth  
  inflating: math-meme-corrector/checkpoint-595/adapter_config.json  
  inflating: math-meme-corrector/checkpoint-595/trainer_state.json  
   creating: math-me

In [2]:
!pip install transformers
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 5.5 MB/s eta 0:00:00


In [3]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("deepseek-ai/deepseek-math-7b-rl")
model = PeftModel.from_pretrained(model, "./math-meme-corrector")

model = model.merge_and_unload()
model.save_pretrained("FixerX_deepseekmath-r1")

/usr/local/lib/python3.11/dist-packages/torch_xla/__init__.py:253: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/626 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.8k [00:00<?, ?B/s]

model-00001-of-000002.safetensors:   0%|          | 0.00/8.59G [00:00<?, ?B/s]

model-00002-of-000002.safetensors:   0%|          | 0.00/5.23G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

In [4]:
!zip -r FixerX_deepseekmath-r1_model.zip ./FixerX_deepseekmath-r1

  adding: FixerX_deepseekmath-r1/ (stored 0%)
  adding: FixerX_deepseekmath-r1/model-00003-of-00006.safetensors (deflated 46%)
  adding: FixerX_deepseekmath-r1/generation_config.json (deflated 25%)
  adding: FixerX_deepseekmath-r1/model-00004-of-00006.safetensors (deflated 46%)
  adding: FixerX_deepseekmath-r1/config.json (deflated 50%)
  adding: FixerX_deepseekmath-r1/model.safetensors.index.json (deflated 95%)
  adding: FixerX_deepseekmath-r1/model-00006-of-00006.safetensors (deflated 51%)
  adding: FixerX_deepseekmath-r1/model-00005-of-00006.safetensors (deflated 46%)
  adding: FixerX_deepseekmath-r1/model-00001-of-00006.safetensors (deflated 48%)
  adding: FixerX_deepseekmath-r1/model-00002-of-00006.safetensors (deflated 45%)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


base_model_name = "deepseek-ai/deepseek-math-7b-rl"
adapter_model_name = "./math-meme-corrector"
merged_model_path = "./FixerX_deepseekmath-r1"


tokenizer = AutoTokenizer.from_pretrained(adapter_model_name)


device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(merged_model_path).to(device)


test_cases = [
    "8 ÷ 2(2+2) = 1",
    "5² = 10",
    "3/6 = 3/2",
    "-3² = 9",
    "0! = 0",
    "50% of 200 = 25",
    "log(a × b) = log(a) + log(b)",
    "√(-1) = -1",
    "(a + b)² = a² + b²",
    "2 + 2 × 2 = 8"
]


def correct_math_meme(math_statement):
    prompt = f"Incorrect: {math_statement}\nCorrect:"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=100, temperature=0.7, do_sample=True)

    corrected_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected_output.split("Correct:")[-1].strip()



print("🔍 Testing Fine-Tuned Model on Incorrect Math Memes:\n")
for i, case in enumerate(test_cases, 1):
    corrected = correct_math_meme(case)
    print(f"❌ Incorrect: {case}\n✅ Model's Correction: {corrected}\n{'-'*50}")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


model_path = "./FixerX_deepseekmath-r1"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)


test_cases = [
    "8 ÷ 2(2+2) = 1",
    "5² = 10",
    "3/6 = 3/2",
    "-3² = 9",
    "0! = 0",
    "50% of 200 = 25",
    "log(a × b) = log(a) + log(b)",
    "√(-1) = -1",
    "(a + b)² = a² + b²",
    "2 + 2 × 2 = 8"
]


def correct_math_meme(math_statement):
    prompt = f"Incorrect: {math_statement}\nCorrect:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=100, temperature=0.7, do_sample=True)

    corrected_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected_output.split("Correct:")[-1].strip()


print("🔍 Testing Fine-Tuned Model on Incorrect Math Memes:\n")
for i, case in enumerate(test_cases, 1):
    corrected = correct_math_meme(case)
    print(f"❌ Incorrect: {case}\n✅ Model's Correction: {corrected}\n{'-'*50}")


In [6]:
!pip install kagglehub

In [11]:
import kagglehub

kagglehub.login()




Kaggle credentials set.
Kaggle credentials successfully validated.


In [12]:
LOCAL_MODEL_DIR = 'FixerX_deepseekmath-r1_model.zip'

kagglehub.model_upload(
  handle = 'buzzgrewal/fixerx/transformers/default',
  local_model_dir = LOCAL_MODEL_DIR,
  version_notes = 'Update 2025-03-07')

Uploading Model https://www.kaggle.com/models/buzzgrewal/fixerx/transformers/default ...
Starting upload for file FixerX_deepseekmath-r1_model.zip


Uploading: 100%|██████████| 14.8G/14.8G [02:06<00:00, 117MB/s]

Upload successful: FixerX_deepseekmath-r1_model.zip (14GB)


Your model instance version has been created.
Files are being processed...
See at: https://www.kaggle.com/models/buzzgrewal/fixerx/transformers/default


In [ ]:
from google.colab import files
files.download("./FixerX_deepseekmath-r1_model.zip")

In [ ]:
!cp "./FixerX_deepseekmath-r1_model.zip" "deepseek_model"

In [ ]:
!pip install google-colab
!pip install googleapiclient

In [ ]:

filename = "FixerX_deepseekmath-r1_model.zip"
folders_or_files_to_save = "deepseek_model"
from google.colab import files
from google.colab import auth
from googleapiclient.http import MediaFileUpload
from googleapiclient.discovery import build

def save_file_to_drive(name, path):
    file_metadata = {
    'name': name,
    'mimeType': 'application/octet-stream'
    }

    media = MediaFileUpload(path,
                  mimetype='application/octet-stream',
                  resumable=True)

    created = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()

    print('File ID: {}'.format(created.get('id')))

    return created



auth.authenticate_user()
drive_service = build('drive', 'v3')

destination_name = zip_file
path_to_file = zip_file
save_file_to_drive(destination_name, path_to_file)